# Iterators and generators

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*. Original material (2020) in collaboration with
*G. Martínez-Lema* and *M. Kekic*.

**Objectives**

* understand what a `for` loop really does: **iterables** and **iterators**;
* write **generators** with `yield`, and know when they save memory;
* read data files **lazily**, one line at a time;
* chain generators into a **pipeline**, the functional way to process data;
* use the iterators of `itertools`.

In [ ]:
import math
import random
import sys
from itertools import islice

## 1. Iterators

An **iterator** is an object that produces a stream of items, one at a time:
`next(it)` gives the next item.

An **iterable** is an object that can give an iterator, with `iter(x)`: lists, tuples,
strings, dictionaries, sets, `range`, files…

In [ ]:
xs = [10, 20, 30]
it = iter(xs)
print(type(it))
print(next(it), next(it), next(it))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — One more</b></p>

The list had three items and we called `next` three times. **Decide**: what does a
fourth `next(it)` do?

</div>

In [ ]:
next(it)

At the end, the iterator raises `StopIteration`. Every iterator is also iterable:
`iter(it)` returns `it` itself.

### 1.1 What a for loop does

A `for` loop is this, written for you:

In [ ]:
xs = [10, 20, 30]

print('for:')
for x in xs:
    print(x)

print('while:')
it = iter(xs)                # what the for does
while True:
    try:
        x = next(it)
    except StopIteration:
        break
    print(x)

That is why `for` works on any iterable: strings, dictionaries (over the keys), files
(over the lines), iterators and generators… and on any object of your own that follows
the protocol: `__iter__` returns an iterator, `__next__` gives the next item.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Twice</b></p>

**Before running**, what do the two `print` show?

```python
squares = map(lambda x: x * x, [1, 2, 3])
print(list(squares))
print(list(squares))
```

</div>

In [ ]:
squares = map(lambda x: x * x, [1, 2, 3])
print(list(squares))
print(list(squares))

<details>
<summary><b>Solution</b></summary>

`[1, 4, 9]` and `[]`. `map` (like `filter`, `zip` and `enumerate`) returns an
**iterator**, not a list. An iterator is consumed as it is read: the first `list` empties
it, and the second one finds nothing left. A list, instead, is an iterable that gives a
**new** iterator every time you loop over it.

This is a classic silent bug: the second loop over an iterator simply does nothing.

`range`, instead, is iterable but **not** an iterator: it does not store its numbers, it
computes them when asked, and it can be walked many times.

</details>

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — How big is range(10**12)?</b></p>

A list of $10^{12}$ integers would need at least 8 TB. **Decide**: do these lines work, and
are they fast?

```python
r = range(10**12)
len(r)
10**11 in r
r[-1]
```

</div>

In [ ]:
r = range(10**12)
print(len(r), 10**11 in r, r[-1], sys.getsizeof(r), 'bytes')

<details>
<summary><b>Solution</b></summary>

All of them work, instantly, and `r` takes 48 bytes. `range` stores only `start`, `stop`
and `step`, and **computes** the answer: the length is `(stop - start) // step`, and
`x in r` is arithmetic, not a search. Lazy objects can be far larger than memory.

</details>

## 2. Generators

A **generator** is a function that uses `yield` instead of `return`. Calling it returns
an iterator.

In [ ]:
def countdown(n):
    """Yield n, n-1, ..., 1."""
    print('start')
    while n > 0:
        yield n
        n -= 1
    print('end')

it = countdown(3)
print(type(it))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — What does it print, and when?</b></p>

`countdown(3)` was called. **Decide** what is printed by that call and by each of the next
four `next(it)`. When are `start` and `end` printed?

</div>

In [ ]:
print(next(it))
print(next(it))
print(next(it))

In [ ]:
next(it)

Calling `countdown(3)` does not run its body. Each `next` runs it **until the next
`yield`**, returns (yields) the value and *pauses* there, keeping its local variables.
The body runs **lazily**, only when the next item is asked for: `end` is printed by the
fourth `next`, just before `StopIteration`. In a `for`, all this is invisible:

In [ ]:
for i in countdown(3):
    print(i)

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — Spot the problem</b></p>

This generator gives the even numbers below `n`. It works. What is wrong with it?

```python
def evens(n):
    """Yield the even numbers in [0, n)."""
    values = [i for i in range(n) if i % 2 == 0]
    for v in values:
        yield v
```

</div>

<details>
<summary><b>Solution</b></summary>

It builds the **whole list first**, and only then yields its items one by one: it uses as
much memory as the list. `evens(10**9)` would need some 20 GB (possibly a `MemoryError`)
before yielding the first number. The point of a generator is to compute each item when it is needed:

```python
def evens(n):
    for i in range(0, n, 2):
        yield i
```

</details>

### 2.1 Why lazy matters: memory

A list stores all its items; a generator stores only its state.
`(x * x for x in range(n))` is a **generator expression**: like a list comprehension,
but with `()`, and lazy.

In [ ]:
n = 1_000_000
as_list = [x * x for x in range(n)]
as_generator = (x * x for x in range(n))

print(sys.getsizeof(as_list), 'bytes')
print(sys.getsizeof(as_generator), 'bytes')

Same result, `sum(as_list) == sum(as_generator)`, but the generator never holds more than
one number, whatever `n` is. And `getsizeof` counts only the list's pointers, not the `int`
objects: the real gap is even larger. This is what you want when you read a 10 GB file, or loop over the
$10^9$ events of an experiment: **one at a time**.

### 2.2 Infinite generators

A generator does not need to end. Whoever reads it decides when to stop:

In [ ]:
def fibonacci():
    """Yield the Fibonacci numbers, forever."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

for f in fibonacci():
    if f > 100:
        break
    print(f, end=' ')

Or let `islice` do the cut: `islice(it, n)`, from `itertools`, gives the first `n` items of
any iterator, like `xs[:n]` for a list (more tools in section 5).

In [ ]:
print(list(islice(fibonacci(), 12)))

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A random walk</b></p>

Write a generator `random_walk(step=1.)` that yields, forever, the position of a 1D
random walk starting at 0: at each step it moves `+step` or `-step` with equal
probability.

1. Take the first 1000 positions (use a `for` with `break`, or `islice(walk, 1000)`).
2. Write a function `first_passage(walk, d)` that returns how many steps the walk needs to
   reach $|x| \ge d$. Repeat it many times: how does the mean grow with $d$?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
from itertools import islice

def random_walk(step=1.):
    """Yield the positions of a 1D random walk, forever."""
    x = 0.
    while True:
        yield x
        x += random.choice((-step, step))

def first_passage(walk, d):
    """Number of steps until |x| >= d."""
    for n, x in enumerate(walk):
        if abs(x) >= d:
            return n

positions = list(islice(random_walk(), 1000))
print(positions[:5])
for d in (5, 10, 20):
    mean = sum(first_passage(random_walk(), d) for _ in range(500)) / 500
    print(d, mean)
```

About 25, 100 and 400: for steps of $\pm 1$ the mean is exactly $d^2$, since the walk
moves a typical distance $\sqrt{n}$ in $n$ steps. This is diffusion.

</details>

## 3. Reading files lazily

A file opened for reading is an **iterator over its lines**. A `for` loop reads one line
at a time: a file of 10 GB can be processed with the memory of one line.

`with open(...) as f:` closes the file when the block ends, even if there is an error.

In [ ]:
# write a small data file: event id, energy (GeV), number of tracks
rng = random.Random(1)
with open('events.txt', 'w') as f:
    f.write('# id energy ntracks\n')
    for i in range(1000):
        f.write(f'{i} {rng.expovariate(1 / 50.):.3f} {rng.randint(0, 6)}\n')

In [ ]:
with open('events.txt') as f:
    print(type(f))
    for line in islice(f, 4):
        print(line.rstrip())

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Reading twice</b></p>

**Before running**, how many lines does each `sum` count?

```python
with open('events.txt') as f:
    print(sum(1 for line in f))
    print(sum(1 for line in f))
```

</div>

In [ ]:
with open('events.txt') as f:
    print(sum(1 for line in f))
    print(sum(1 for line in f))

<details>
<summary><b>Solution</b></summary>

`1001` and `0`: a file is its own iterator, and the first loop reaches the end. To read
it again, open it again (or `f.seek(0)`).

</details>

### 3.1 A generator that parses

A generator turns lines of text into data, one event at a time, and hides the details of
the format (comments, columns, types):

In [ ]:
def read_events(path):
    """Yield the events of a file as dicts: id, energy (GeV), ntracks."""
    with open(path) as f:
        for line in f:
            if not line.strip() or line.startswith('#'):
                continue
            i, energy, ntracks = line.split()
            yield {'id': int(i), 'energy': float(energy), 'ntracks': int(ntracks)}

print(next(read_events('events.txt')))

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — One pass over a file</b></p>

Using `read_events`, and reading the file **only once**, compute the number of events,
the mean energy and the maximum energy. You may not store the events in a list. Then
check the result against a version that does load everything with a list.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def summary(events):
    """Number of events, mean and maximum energy, in one pass."""
    n, total, emax = 0, 0., -math.inf
    for e in events:
        n += 1
        total += e['energy']
        emax = max(emax, e['energy'])
    return n, total / n, emax

n, mean, emax = summary(read_events('events.txt'))

energies = [e['energy'] for e in read_events('events.txt')]     # the list version
assert n == len(energies)
assert math.isclose(mean, sum(energies) / n)
assert emax == max(energies)
```

`sum` and `max` alone would each need their own pass: an iterator can be read only once.
Computing several things in one pass is the price, and the point, of streaming.

</details>

## 4. Pipelines

Generators can be **chained**: each one reads from the previous one and yields to the
next. The data flows through the chain one item at a time: a functional style of
processing data.

A toy experiment: a source of events, a selection, and a computed quantity.

In [ ]:
def events(n, seed=1):
    """Yield n toy events: dicts with the energy (GeV) and the number of tracks."""
    rng = random.Random(seed)
    for i in range(n):
        yield {'id': i, 'energy': rng.expovariate(1 / 50.), 'ntracks': rng.randint(0, 6)}

def select(evts, emin):
    """Yield the events with energy above emin."""
    for e in evts:
        if e['energy'] > emin:
            yield e

def energy_per_track(evts):
    """Yield energy / ntracks for the events with tracks."""
    for e in evts:
        if e['ntracks'] > 0:
            yield e['energy'] / e['ntracks']

In [ ]:
pipeline = energy_per_track(select(events(100_000), emin=100.))
values = list(pipeline)
print(len(values), sum(values) / len(values))

The source can be anything that yields events: a simulation, or the file of section 3.
The rest of the chain does not change:

In [ ]:
values = list(energy_per_track(select(read_events('events.txt'), emin=100.)))
print(len(values))

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Why not lists?</b></p>

The same could be done with three functions that take and return lists. What do we gain
with generators? What do we lose? (Think of: memory, reading the data twice, debugging.)

</div>

<details>
<summary><b>Solution</b></summary>

**Gain**: memory (only one event is alive at a time), and the first result comes out
immediately; each stage is a small function with a single job, testable on its own.

**Cost**: the pipeline can be read **only once** (section 1); there is no `len`, no
indexing; and when something fails, it fails in the middle of the chain. To debug,
materialize a small piece: `list(islice(select(events(100), 100.), 5))`.

</details>

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Where did the events go?</b></p>

**Before running**, what does this print? Why?

```python
selected = select(events(1000), emin=100.)
n = sum(1 for _ in selected)
emax = max((e['energy'] for e in selected), default=None)
print(n, emax)
```

</div>

In [ ]:
selected = select(events(1000), emin=100.)
n = sum(1 for _ in selected)
emax = max((e['energy'] for e in selected), default=None)
print(n, emax)

<details>
<summary><b>Solution</b></summary>

`n` is right (about $1000\,e^{-2} \approx 135$), and `emax` is `None`: counting consumed the
generator, and `max` found it empty. Without `default=None` it would fail with a
`ValueError` that points far from the cause. Either compute both in one pass, or
build the pipeline again.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (in pairs) — Build the chain</b></p>

In pairs. Student A adds a stage that **smears** the energy with a 5 % Gaussian
resolution; student B adds a stage that yields only the first `k` events. Write each stage as a generator,
with a test on a small input. Then plug both into the pipeline. Does the order of the
stages matter?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def smear(evts, resolution=0.05, seed=2):
    rng = random.Random(seed)
    for e in evts:
        yield {**e, 'energy': e['energy'] * rng.gauss(1., resolution)}

def first(evts, k):
    for i, e in enumerate(evts):
        if i >= k:
            return
        yield e

# tests on small inputs
assert len(list(first(events(10), 3))) == 3
assert all(e['ntracks'] == f['ntracks'] for e, f in zip(events(5), smear(events(5))))
e0 = next(events(1))
list(smear([e0]))
assert e0 == next(events(1))         # smear does not modify its input

values = list(energy_per_track(select(smear(events(100_000)), emin=100.)))
```

`smear` returns a **new** dict, `{**e, ...}`: it does not mutate its input. The order
matters: smearing before or after the selection gives different samples, and only
*before* is the physics of a real detector. And `first` before `select` gives `k`
generated events, after it `k` selected ones. `first` is `itertools.islice(evts, k)`.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (in pairs) — Producer and consumer</b></p>

In pairs, agreeing only on the format of the items: a tuple `(t_B, t_C)`.

* **A** writes the **producer**: a generator of decay chains $A \to B \to C$. Each item is
  the time at which $B$ appears and the time at which $C$ appears, with exponential
  lifetimes $\tau_A = 2$ and $\tau_B = 5$ (`random.expovariate`).
* **B** writes the **consumer**, without seeing A's code: a function that reads $n$ items
  from any iterator and estimates $\tau_A$ and $\tau_B$.

Connect them. Do the estimates agree with the inputs within their statistical errors,
$\tau/\sqrt{n}$, within a few $\sigma$?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def decay_chains(tau_a, tau_b, seed=1):
    """Yield (t_B, t_C), the times at which B and C appear, forever."""
    rng = random.Random(seed)
    while True:
        t_b = rng.expovariate(1 / tau_a)
        yield t_b, t_b + rng.expovariate(1 / tau_b)

def estimate_lifetimes(chains, n):
    """Mean lifetimes of A and B from the first n chains."""
    sum_a = sum_b = 0.
    k = 0
    for k, (t_b, t_c) in enumerate(islice(chains, n), start=1):
        sum_a += t_b
        sum_b += t_c - t_b
    if k < n:
        raise ValueError(f'only {k} chains, {n} were asked')
    return sum_a / n, sum_b / n

n = 10_000
tau_a, tau_b = estimate_lifetimes(decay_chains(2., 5.), n)
assert abs(tau_a - 2.) < 5 * 2. / math.sqrt(n)
assert abs(tau_b - 5.) < 5 * 5. / math.sqrt(n)
```

The consumer never knows that the producer is a simulation: a file of real chains, read
with a generator, would work the same. Agreeing on the **interface** (here, the tuple)
is what lets two people work at the same time.

Why the check on `k`? `islice` stops silently when the iterator runs out. A file with
500 chains and `n = 1000` would give lifetimes **half** the true ones, with no error. A
consumer must not trust the producer to have enough items.

</details>

## 5. itertools

The module `itertools` has the iterators you would otherwise write yourself. A few:

| | |
|:--|:--|
| `islice(it, n)` | the first `n` items of any iterator |
| `count(start)` | `start, start+1, …`, forever |
| `chain(a, b)` | `a` and then `b` |
| `product(a, b)` | all pairs $(a_i, b_j)$, as two nested `for` |
| `combinations(xs, 2)` | all the pairs without repetition |
| `accumulate(xs)` | running sums: `x0, x0+x1, x0+x1+x2, …` |
| `groupby(xs, key)` | groups of **consecutive** items with the same key |

In [ ]:
from itertools import accumulate, groupby

print(list(accumulate([1, 2, 3, 4])))
runs = [(1, 5.), (1, 7.), (2, 3.), (2, 4.), (1, 9.)]
print([(k, len(list(g))) for k, g in groupby(runs, key=lambda r: r[0])])

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Invariant mass of all pairs</b></p>

An event has these particles, as `(E, px, py, pz)` in GeV. Compute the invariant mass
$m^2 = (E_1 + E_2)^2 - |\mathbf{p}_1 + \mathbf{p}_2|^2$ of every **pair** with
`itertools.combinations`, and find the pair closest to the $Z$ mass, 91.19 GeV
(toy data: the closest is not exact).

```python
particles = [(50.2, 10.1, -3.2, 49.0), (45.1, -9.8, 2.9, -43.9),
             (12.0, 1.0, 11.0, 4.6), (30.5, -2.0, -20.1, 22.8)]
```

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
from itertools import combinations

particles = [(50.2, 10.1, -3.2, 49.0), (45.1, -9.8, 2.9, -43.9),
             (12.0, 1.0, 11.0, 4.6), (30.5, -2.0, -20.1, 22.8)]

def mass(p1, p2):
    e, px, py, pz = (a + b for a, b in zip(p1, p2))
    return math.sqrt(max(e * e - px * px - py * py - pz * pz, 0.))

pairs = list(combinations(range(len(particles)), 2))
best = min(pairs, key=lambda ij: abs(mass(particles[ij[0]], particles[ij[1]]) - 91.19))
print(best, mass(particles[best[0]], particles[best[1]]))
```

With $n$ particles there are $n(n-1)/2$ pairs: `combinations` avoids both the pair
`(i, i)` and the repeated `(j, i)`.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A running mean</b></p>

Estimate $\pi$ by Monte Carlo (a point in the unit square falls inside the quarter circle
with probability $\pi/4$). With a generator of 0/1 hits and `itertools.accumulate`, compute
the **running estimate** after each point, and print it at $n = 10, 100, \ldots, 10^5$.
Does the error shrink as $1/\sqrt{n}$?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
from itertools import accumulate

def hits(seed=1):
    rng = random.Random(seed)
    while True:
        yield 1 if rng.random() ** 2 + rng.random() ** 2 < 1. else 0

checkpoints = {10, 100, 1_000, 10_000, 100_000}
for n, inside in enumerate(accumulate(islice(hits(), 100_000)), start=1):
    if n in checkpoints:
        estimate = 4 * inside / n
        print(f'{n:>7} {estimate:.5f} {abs(estimate - math.pi):.5f}')
```

Each factor 10 in $n$ should reduce the error by about $\sqrt{10} \approx 3$ (a factor 100
by 10), with large fluctuations: one run is one sample of the error.

</details>

***[+] Lookout.*** `groupby` only groups **consecutive** items: sort by the same key first, or you get the
same key in several groups. `tee(it, 2)` splits an iterator in two. A generator can also
**receive** values: `it.send(value)` resumes it and makes
`value` the result of the `yield` expression (prime it first with `next(it)`), and
`it.close()` stops it. This turns
generators into *coroutines*, the ancestors of `async`/`await`. And `yield from other`
delegates to another generator. See the
[Functional Programming HOWTO](https://docs.python.org/3/howto/functional.html#passing-values-into-a-generator).

## 6. Review

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — Review this generator</b></p>

It should yield the energies above `emin` of a list of events. It does not. Find the two
bugs before running it.

```python
def high_energies(evts, emin):
    for e in evts:
        if e['energy'] > emin:
            return e['energy']
    yield
```

</div>

In [ ]:
def high_energies(evts, emin):
    for e in evts:
        if e['energy'] > emin:
            return e['energy']
    yield

print(list(high_energies([{'energy': 5.}, {'energy': 150.}, {'energy': 200.}], 100.)))

<details>
<summary><b>Solution</b></summary>

1. `return` instead of `yield`: in a generator, `return` **ends** the iteration, so the
   first high energy stops it without yielding anything: the result above is `[]`.
2. The final `yield` (with no value) is what makes the function a generator; with no
   high energy at all, it yields a lone `None`.

```python
def high_energies(evts, emin):
    for e in evts:
        if e['energy'] > emin:
            yield e['energy']
```

Or, in one line, a generator expression: `(e['energy'] for e in events if e['energy'] > emin)`.

</details>

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Four ways to add squares</b></p>

$\sum_{k < n} k^2$ for $n = 10^7$:

```python
total = 0
for k in range(n):
    total += k * k

total = sum([k * k for k in range(n)])
total = sum(k * k for k in range(n))
total = sum(map(lambda k: k * k, range(n)))
```

Which ones build a list of $10^7$ numbers? Which one would you write? And how would numpy
do it? Time them with `%timeit` if you are curious.

</div>

<details>
<summary><b>Solution</b></summary>

Only the second builds the list (about 80 MB of pointers plus about 300 MB of `int`
objects); the others
produce one number at a time. The third is the idiomatic one. With numpy,
`(np.arange(n) ** 2).sum()` is much faster, but it builds an array of $10^7$ numbers,
and here it gives a **wrong** answer: the exact sum, `(n - 1) * n * (2 * n - 1) // 6`, is
about $3.3 \times 10^{20}$, beyond the largest `int64`, $9.2 \times 10^{18}$, and numpy
overflows silently. Python integers never
overflow. (With floats, `np.arange(n, dtype=float)`, it works, with rounding.)

</details>

## 7. Summary

* Iterators give the items one by one; generators are the easy way to write them.
* Files are read lazily, line by line; a generator hides the format.
* A pipeline chains source → selection → computation, one item at a time.
* `itertools` has the iterators you would otherwise write.

<details>
<summary><b>[i] Cheat sheet — Iterators and generators</b></summary>

| | |
|:--|:--|
| iterable | can give an iterator: `iter(x)`. Lists, strings, dicts, `range`, files |
| iterator | `next(it)` gives the next item; `StopIteration` at the end; consumed when read |
| `for x in xs` | `it = iter(xs)` + `next(it)` until `StopIteration` |
| generator function | a `def` with `yield`; calling it returns an iterator |
| `(f(x) for x in xs)` | generator expression |
| lazy | each item is computed when asked: constant memory, infinite streams |
| `with open(p) as f: for line in f` | a file is an iterator over its lines |
| pipeline | `c(b(a(source)))`: generators chained, one item at a time |
| `itertools` | `islice`, `count`, `chain`, `product`, `combinations`, `accumulate`, `groupby` |

**Traps**: reading an iterator twice (`map`, `filter`, `zip`, files, generators); a
generator that builds a list first; `return` instead of `yield`.

</details>

### Check yourself

1. What is the difference between an iterable and an iterator? Give an example of each.
2. What does `next` do on an iterator that has no more items?
3. When does the body of a generator function start to run?
4. `z = zip(xs, ys)`; you loop over `z` twice. What happens the second time?
5. Why can a generator handle a file larger than the computer's memory?
6. Does `10**11 in range(10**12)` need a search?
7. Why does `sum(1 for _ in f)` give 0 the second time over the same open file?

<details>
<summary><b>Solutions</b></summary>

1. An iterable can give an iterator (`[1, 2]`, `range(3)`); an iterator gives the items
   one by one and is consumed (`iter([1, 2])`, `map(...)`, a generator).
2. It raises `StopIteration`.
3. At the first `next`, not when the function is called.
4. Nothing: `zip` returns an iterator, already consumed by the first loop.
5. It holds one item at a time, not the whole file.
6. No: `range` computes the answer with arithmetic.
7. A file is its own iterator: the first loop reached the end.

</details>

### Bibliography

1. H. Abelson, G. J. Sussman and J. Sussman, *Structure and Interpretation of Computer
   Programs*, 2nd ed., MIT Press (1996), section 3.5, *Streams*.
2. [Functional Programming HOWTO](https://docs.python.org/3/howto/functional.html),
   Python documentation.
3. [itertools](https://docs.python.org/3/library/itertools.html), Python documentation.